# 05. Universe Selection & Strategy Construction

## 📋 개요
모델의 예측 결과를 바탕으로 **수익률, 정확도, 위험도를 종합 평가하여 최적의 투자 후보군(Universe)**을 선정합니다.

## ✨ 핵심 전략 및 최근 업데이트
- **v3.10.0**: `directional_accuracy`(방향성 정확도) 리포트 컬럼 추가. `Top-k Precision` 계산 셀 신설.
- **v3.9.2**: `리스크점수` 제거 → 5개 위험 지표 개별 컬럼 전환. `filter_statistics.json` 저장 추가.
- **Facade Pattern**: `select_investment_universe()` 단일 함수로 복잡한 평가 로직 캡슐화.
- **사다리꼴 보정 앵커 (v3.9.1)**: `log_return_1d` 타겟 모드 사용 시 실측 등락률을 앵커로 주입.
- **비현실적 수익률 필터 (v3.7.2)**: `strategy.max_daily_return` 설정으로 급등 시나리오 자동 제외.

## 🔄 데이터 흐름
```text
[과거 예측 (test_predictions)] ──┐
[미래 예측 (future_forecasts)] ──┼─→ [ select_investment_universe() ]
[메타 데이터 (dataset.parquet)] ─┘         ├─ 전략 파라미터 적용
                                           ├─ 평가 (정확도·수익성·위험도)
                                           └─ Top-K 후보 선정
                                                  ↓
                                    [ 투자 후보군 (CSV / Parquet) ]
```

## 🔧 Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings

from src.utils.config import load_config, ProjectPaths
from src.universe.select_universe import select_investment_universe

warnings.filterwarnings('ignore')

## 1️⃣ 설정 및 경로 초기화

In [ ]:
cfg = load_config()
paths = ProjectPaths.from_config(cfg)
paths.ensure_dirs()

MODEL_DATE = cfg['universe']['model_date']

print(f"📅 날짜 기준 설정:")
print(f"   - 프로젝트 기준일: {cfg['project']['reference_date']}")
print(f"   - 모델 학습 기준일: {MODEL_DATE}")
print(f"\n📁 경로:")
print(f"   - 과거 예측: {paths.get_predictions_parquet()}")
print(f"   - 미래 예측: {paths.get_forecasts_parquet()}")
print(f"   - Universe 출력: {paths.universe_dir}")

## 2️⃣ 데이터 로드

In [ ]:
print("\n📥 데이터 로드 중...")

past_pred_path = paths.training_dir / "test_predictions.parquet"
df_past_pred = pd.read_parquet(past_pred_path)
df_past_pred['date'] = pd.to_datetime(df_past_pred['date'])

print(f"\n[과거 예측]")
print(f"   - 파일: {past_pred_path}")
print(f"   - 행수: {len(df_past_pred):,}")
print(f"   - 기간: {df_past_pred['date'].min()} ~ {df_past_pred['date'].max()}")
print(f"   - 종목 수: {df_past_pred['ticker'].nunique()}")

future_pred_path = paths.get_forecasts_parquet()
df_future = pd.read_parquet(future_pred_path)
df_future['date'] = pd.to_datetime(df_future['date'])

print(f"\n[미래 예측]")
print(f"   - 파일: {future_pred_path}")
print(f"   - 행수: {len(df_future):,}")
print(f"   - 기간: {df_future['date'].min()} ~ {df_future['date'].max()}")
print(f"   - 종목 수: {df_future['ticker'].nunique()}")

dataset_path = paths.get_dataset_parquet()
df_meta = pd.read_parquet(dataset_path)
df_meta['date'] = pd.to_datetime(df_meta['date'])

latest_meta_date = df_meta['date'].max()
df_meta_latest = df_meta[df_meta['date'] == latest_meta_date].copy()

print(f"\n[메타 데이터]")
print(f"   - 파일: {dataset_path}")
print(f"   - 기준일: {latest_meta_date}")
print(f"   - 종목 수: {df_meta_latest['ticker'].nunique()}")

print("\n✅ 데이터 로드 완료")

## 3️⃣ Universe 선정 실행 (Facade Pattern)

In [ ]:
print("\n" + "="*65)
print("3️⃣ Universe 선정 실행 (Facade Pattern)")
print("="*65)

strategy_cfg     = cfg.get('strategy', {})
MIN_HOLD_DAYS    = strategy_cfg.get('min_hold_days', 5)
MAX_DAILY_RETURN = strategy_cfg.get('max_daily_return', 0.16)
TOP_K            = strategy_cfg.get('top_k', 200)

train_cfg   = cfg.get('training', {})
target_base = train_cfg.get('target_col_name', 'target_log_close')
horizons    = train_cfg.get('horizons', [1, 2, 3, 4, 5])
TARGET_COLS = [f'{target_base}_h{h}' for h in horizons]

target_type = train_cfg.get('target_type', 'log_close')

if target_type == 'log_return_1d':
    ref_cols = ['ticker', 'date', 'target_log_close']
    if 'target_log_return_1d' in df_meta.columns:
        ref_cols.append('target_log_return_1d')
    LOG_CLOSE_REF = df_meta[ref_cols].copy()
else:
    LOG_CLOSE_REF = None

print(f"\n[전략 파라미터]")
print(f"   - 최소 보유 기간:  {MIN_HOLD_DAYS}일")
print(f"   - 수익률 상한:     일평균 {MAX_DAILY_RETURN:.1%}")
print(f"   - 평가 타겟 컬럼:  {TARGET_COLS[0]} ~ {TARGET_COLS[-1]}")
print(f"   - 로그 종가 환산:  {'사다리꼴 앵커 적용' if LOG_CLOSE_REF is not None else '미적용 (log_close 모드)'}")

results = select_investment_universe(
    df_past_predictions=df_past_pred,
    df_future_forecasts=df_future,
    df_meta=df_meta_latest,
    model_date=MODEL_DATE,
    top_k=TOP_K,
    min_hold_days=MIN_HOLD_DAYS,
    max_daily_return=MAX_DAILY_RETURN,
    target_columns=TARGET_COLS,
    log_close_ref=LOG_CLOSE_REF,
    verbose=True,
)

df_accuracy      = results['accuracy']
df_return        = results['returns']
df_risk          = results['risk']
df_full_universe = results['full']
df_candidates    = results['candidates']
filter_stats     = results['filter_stats']

## 4️⃣ 사용자 선택을 위한 상세 리포트 생성

선정된 후보군(`df_candidates`)을 사용자가 읽기 쉽도록 한글 컬럼명으로 정리합니다.
- **수익성 지표**: 예상총수익률(%), 예상일평균수익률, 최적보유기간
- **정확도 지표**: IC(상관계수), 신뢰도(RMSE 역수), RMSE, 방향성정확도 (✨ v3.10.0)
- **위험 지표**: 변동성, 하방위험, VaR(95%), CVaR(95%), 최대낙폭(MDD)
- **매매 가이드**: 목표 매수일/매도일 및 매수가/매도가

In [ ]:
print("\n" + "="*65)
print("4️⃣ 투자 후보 상세 리포트 생성")
print("="*65)

try:
    master_path = paths.get_ticker_master()
    df_master = pd.read_csv(master_path)
    ticker_name_map = dict(zip(df_master['ticker'].astype(str), df_master['name']))
    df_candidates['종목명'] = df_candidates['ticker'].map(ticker_name_map)
    print("✅ ticker_master 로드 완료")
except Exception as e:
    print(f"⚠️  ticker_master 로드 실패: {e}")
    df_candidates['종목명'] = df_candidates['ticker']

df_candidates_report = df_candidates.copy()

df_candidates_report['순위'] = df_candidates_report['return_rank']
df_candidates_report['종목코드'] = df_candidates_report['ticker']

df_candidates_report['예상일평균수익률(로그)'] = df_candidates_report['daily_log_return'].round(6)
df_candidates_report['예상총수익률(%)'] = df_candidates_report['total_return_pct'].round(2)
df_candidates_report['최적보유기간(일)'] = df_candidates_report['hold_days'].astype(int)

df_candidates_report['IC'] = df_candidates_report['ic_mean'].round(4)
df_candidates_report['신뢰도(RMSE역수)'] = df_candidates_report['confidence_rmse'].round(4)
df_candidates_report['RMSE'] = df_candidates_report['rmse'].round(4)
# ✨ v3.10.0: 방향성 정확도 컬럼 추가
df_candidates_report['방향성정확도'] = df_candidates_report['directional_accuracy'].round(4)

df_candidates_report['변동성'] = df_candidates_report['volatility'].round(4)
df_candidates_report['하방위험'] = df_candidates_report['downside_risk'].round(4)
df_candidates_report['VaR(95%)'] = df_candidates_report['var_95'].round(4)
df_candidates_report['CVaR(95%)'] = df_candidates_report['cvar_95'].round(4)
df_candidates_report['최대낙폭(%)'] = (df_candidates_report['max_drawdown'] * 100).round(2)

df_candidates_report['매수일'] = pd.to_datetime(df_candidates_report['buy_date']).dt.strftime('%Y-%m-%d')
df_candidates_report['매도일'] = pd.to_datetime(df_candidates_report['sell_date']).dt.strftime('%Y-%m-%d')
df_candidates_report['매수가'] = df_candidates_report['buy_price'].round(0).astype(int)
df_candidates_report['매도가'] = df_candidates_report['sell_price'].round(0).astype(int)
df_candidates_report['유동성점수'] = df_candidates_report['liquidity_score'].round(0).astype(int)

report_cols = [
    '순위', '종목코드', '종목명',
    '예상일평균수익률(로그)', '예상총수익률(%)', '최적보유기간(일)',
    'IC', '신뢰도(RMSE역수)', 'RMSE', '방향성정확도',
    '변동성', '하방위험', 'VaR(95%)', 'CVaR(95%)', '최대낙폭(%)',
    '매수일', '매도일', '매수가', '매도가',
    '유동성점수'
]

df_report = df_candidates_report[report_cols]

print("\n📊 Top 20 종목 미리보기:")
display_cols_short = [
    '순위', '종목명', '예상총수익률(%)', '최적보유기간(일)',
    'IC', '방향성정확도', '신뢰도(RMSE역수)', '변동성', '하방위험',
    '매수가', '매도가'
]

display(df_report[display_cols_short].head(20))
print("\n✅ 리포트 생성 완료")

## 5️⃣ 결과 저장

In [ ]:
print("\n" + "="*65)
print("5️⃣ 결과 저장")
print("="*65)

full_universe_path = paths.get_universe_full()
df_full_universe.to_parquet(full_universe_path, index=False)
print(f"\n💾 전체 Universe 저장: {full_universe_path}")
print(f"   - 종목 수: {len(df_full_universe):,}")

candidates_path = paths.get_universe_candidates()
df_candidates.to_parquet(candidates_path, index=False)
print(f"\n💾 투자 후보 저장: {candidates_path}")
print(f"   - 종목 수: {len(df_candidates):,}")

report_csv_path = paths.get_investment_report_csv()
df_report.to_csv(report_csv_path, index=False, encoding='utf-8-sig')
print(f"\n💾 상세 리포트 저장: {report_csv_path}")
print(f"   - 형식: CSV (Excel 호환)")
print(f"   - 컬럼 수: {len(report_cols)}개")

import json as _json
filter_stats_path = paths.get_filter_statistics()
with open(filter_stats_path, 'w', encoding='utf-8') as _f:
    _json.dump(filter_stats, _f, indent=2, ensure_ascii=False)
print(f"\n💾 필터링 통계 저장: {filter_stats_path}")

try:
    excel_path = paths.get_investment_report_excel()

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        df_report.head(20).to_excel(writer, sheet_name='Top20', index=False)
        df_report.to_excel(writer, sheet_name='전체후보', index=False)

        df_risk_groups = df_report.copy()
        df_risk_groups['위험등급'] = pd.cut(
            df_risk_groups['변동성'],
            bins=pd.qcut(df_risk_groups['변동성'], q=4, retbins=True)[1],
            labels=['낮음', '보통', '높음', '매우높음']
        )
        for risk_level in ['낮음', '보통', '높음', '매우높음']:
            df_level = df_risk_groups[df_risk_groups['위험등급'] == risk_level]
            if len(df_level) > 0:
                df_level.to_excel(writer, sheet_name=f'위험_{risk_level}', index=False)

    print(f"\n💾 Excel 리포트 저장: {excel_path}")
    print(f"   - Sheet: Top20, 전체후보, 위험_낮음, 위험_보통 등")

except Exception as e:
    print(f"\n⚠️  Excel 저장 실패 (openpyxl 필요): {e}")

print("\n" + "="*65)
print("✅ [Step 5] Universe 선정 완료")
print("="*65)

## 6️⃣ Top-k Precision 계산 (✨ v3.10.0)

**Top-k Precision**: 상위 K개 추천 종목 중 test fold 기간 내 실제 수익이 발생한 비율.
- 기준: `daily_log_return > 0` (일평균 로그 수익률 양수)
- 추가 데이터 수집 불필요 — `test_predictions.parquet`와 `universe_candidates.parquet`만으로 계산

> 💡 **해석 가이드**: Top-k Precision이 높을수록 모델의 종목 선별 능력이 우수합니다. 무작위 선택의 기준선은 전체 후보 중 수익 발생 종목의 비율입니다.

In [ ]:
print("\n" + "="*65)
print("6️⃣ Top-k Precision")
print("="*65)

# 후보 종목 목록 (return_rank 기준 정렬)
top_k_tickers = df_candidates.sort_values('return_rank')['ticker'].tolist()

# test fold 기간 내 실제 수익 발생 여부: daily_log_return > 0
# df_candidates에 이미 포함된 daily_log_return(미래 예측 기반)이 아닌
# test_predictions 기반 실제 성과를 사용
# → df_past_pred에서 종목별 실측 수익률을 산출

# true_cols로 horizon별 실측값 취합
true_col_candidates = [c for c in df_past_pred.columns if c.startswith('true_')]

if true_col_candidates:
    # 종목별 test 기간 내 평균 실측 등락률 산출
    realized = (
        df_past_pred.groupby('ticker')[true_col_candidates]
        .mean()
        .mean(axis=1)  # horizon 평균
        .reset_index()
    )
    realized.columns = ['ticker', 'realized_mean_return']

    # 전체 후보 기준선: 수익 발생 비율
    n_profitable_all = (realized['realized_mean_return'] > 0).sum()
    baseline = n_profitable_all / len(realized) if len(realized) > 0 else np.nan

    # Top-k별 Precision 계산
    k_list = [10, 20, 50, 100, len(top_k_tickers)]
    print(f"\n{'K':>6} | {'Precision':>10} | {'수익종목수':>10} | {'기준선':>8}")
    print("-" * 42)
    for k in k_list:
        top_k = top_k_tickers[:k]
        realized_topk = realized[realized['ticker'].isin(top_k)]
        n_profitable = (realized_topk['realized_mean_return'] > 0).sum()
        precision = n_profitable / len(realized_topk) if len(realized_topk) > 0 else np.nan
        print(f"{k:>6} | {precision:>10.4f} | {n_profitable:>10} | {baseline:>8.4f}")
else:
    print("⚠️  test_predictions에 true_ 컬럼이 없습니다. Top-k Precision을 계산할 수 없습니다.")

print("\n✅ Top-k Precision 계산 완료")

## 📊 전체 후보 요약 통계

In [ ]:
print("\n📊 전체 후보 요약 통계:")

print(f"\n[수익률 분포]")
print(f"   - 10% 이상: {(df_report['예상총수익률(%)'] >= 10).sum()}개")
print(f"   - 5~10%: {((df_report['예상총수익률(%)'] >= 5) & (df_report['예상총수익률(%)'] < 10)).sum()}개")
print(f"   - 0~5%: {((df_report['예상총수익률(%)'] >= 0) & (df_report['예상총수익률(%)'] < 5)).sum()}개")
print(f"   - 음수: {(df_report['예상총수익률(%)'] < 0).sum()}개")

print(f"\n[정확도 분포]")
print(f"   - IC 0.7 이상: {(df_report['IC'] >= 0.7).sum()}개")
print(f"   - IC 0.6~0.7: {((df_report['IC'] >= 0.6) & (df_report['IC'] < 0.7)).sum()}개")
print(f"   - IC 0.5~0.6: {((df_report['IC'] >= 0.5) & (df_report['IC'] < 0.6)).sum()}개")
print(f"   - 방향성정확도 0.6 이상: {(df_report['방향성정확도'] >= 0.6).sum()}개")
print(f"   - 방향성정확도 평균: {df_report['방향성정확도'].mean():.4f}")

print(f"\n[위험도 분포]")
print(f"   - 변동성 중앙값: {df_report['변동성'].median():.4f}")
print(f"   - 하방위험 평균: {df_report['하방위험'].mean():.4f}")
print(f"   - VaR(95%) 평균: {df_report['VaR(95%)'].mean():.4f}")
print(f"   - CVaR(95%) 평균: {df_report['CVaR(95%)'].mean():.4f}")
print(f"   - MDD 평균: {df_report['최대낙폭(%)'].mean():.2f}%")

## 🏁 완료 및 다음 단계

### ✅ 생성된 산출물
1. **`universe_candidates.parquet`**: 시스템이 선정한 Top-K 후보군
2. **`investment_report.csv` / `.xlsx`**: 사용자 검토용 상세 분석 리포트 ⭐
3. **`filter_statistics.json`**: 필터링 단계별 탈락 종목 수 통계

### 🎯 투자 후보 결정 가이드
1. **상위 20위 검토**: 기대 수익률이 가장 높은 종목들
2. **신뢰도 교차 검증**: IC ≥ 60% AND 방향성정확도 참고
3. **리스크 확인**: 변동성, 하방위험, VaR, CVaR, MDD 직접 검토
4. **매매 계획**: 최적보유기간과 매수/매도 목표가 참고

### 🚀 다음 작업
- **06단계**: 포트폴리오 최적화 (MVO 등)
- **백테스트**: 과거 구간 시뮬레이션